## Lab 2 - Part B : Inverse Kinematics

### Overview & Variables

Part B = เอาคำตอบ IK ที่แก้ **ด้วยมือใน Part A** มาตรวจด้วย Robotics Toolbox แล้ว cross-check ด้วย Forward Kinematics (FK)

| Variable | What is it? | From |
|--------|---------|-------------|
| `M` | ตำแหน่งเป้าหมายใน **base frame** ของหุ่น (= `Pobj`) หน่วยเมตร `[x, y, z]` | Part A: เอา `M` ใน camera frame มาแปลงผ่าน `BaseCameraT` (ข้อ 1) → ได้ base frame (ข้อ 4) |
| `q_goal` | มุม/ระยะข้อต่อที่แก้ IK **ด้วยมือ** (เลือกมา 1 branch) | Part A ข้อ 4 |
| `branches` | คำตอบ IK **ทุก branch** | Part A ข้อ 2/3 |

**หน่วยของ `q`** ขึ้นกับชนิดข้อต่อของหุ่นที่ออกแบบใน LAB1: revolute → เรเดียน `[rad]`, prismatic → เมตร `[m]`
(หุ่นใน template นี้เป็นแบบ **RRP** → `q1, q2` = rad, `q3` = m)


### 0. Setup
- Load your robot like LAB1

In [ ]:
import numpy as np
from pathlib import Path
from roboticstoolbox import Robot
from spatialmath import SE3
import spatialgeometry as sg
import swift

HERE = Path.cwd().as_posix()
links, name, _, _ = Robot.URDF_read('my_robot/robot.urdf', tld=HERE)
robot = Robot(links, name=name)
print('loaded', name, robot.n, 'joints')

env = swift.Swift()
env.launch(realtime=True, browser="notebook")
env.add(robot)
robot.q = [0.0, 0.0, 0.0]   # home configuration
env.step()

In [ ]:
# Allowable syntax (reference only - do not run as-is):
# robot.fkine(q)          # forward kinematics -> transform T ; tip position is T.t
# np.linalg.norm(a - b)   # distance between two points
# SE3(*M)                 # make a target pose from a point M = [x, y, z]
# robot.ikine_LM(SE3(*M), q0=..., mask=[1,1,1,0,0,0], joint_limits=True)   # numerical IK -> sol
# sol.q                   # the joint solution it found
# sol.success             # did it converge (True/False)

### 1. target M and q  (from Part A item 4)
hint: fill M (base frame) and q_goal, each is np.array of 3 numbers

**Fill:** `M` และ `q_goal` (ในเซลล์โค้ดถัดไป)
- `M` = ตำแหน่งเป้าหมายใน **base frame** (`Pobj`) จาก **Part A ข้อ 4** — `np.array` 3 ตัว `[x, y, z]` หน่วยเมตร
- `q_goal` = คำตอบ IK ที่**แก้ด้วยมือ**ใน Part A ข้อ 4 (เลือกมา 1 branch) — `np.array` 3 ตัว

**Code:** วางลูกบอลแดงไว้ที่ `M` แล้วสั่งแขนไปที่ `q_goal` → ถ้าคำตอบถูก **ปลายมือควรไปแตะลูกบอลพอดี**

In [ ]:
M = np.array([0.0, 0.0, 0.0])       # edit here
q_goal = np.array([0.0, 0.0, 0.0])  # edit here

if np.allclose(M, 0) or np.allclose(q_goal, 0):
    print('fill in M and q_goal first, then run again')
else:
    marker = sg.Sphere(radius=0.03, color=(1.0, 0.1, 0.1, 0.6))
    marker.T = SE3(*M)
    env.add(marker)
    env.step()
    print('target shown at', np.round(M, 4))

In [ ]:
# move the robot to q_goal  (tip should reach the ball)
robot.q = q_goal
env.step()

### 2. cross-check with FK
hint: robot.fkine(q) -> T, then T.t -> tip (x,y,z), and np.linalg.norm(tip - M) -> error

**Fill:** `tip` และ `dist`
- `tip` = ตำแหน่งปลายมือที่ได้จาก **FK** ของ `q` → เรียก `robot.fkine(q)` ได้ transform `T` แล้วเอา `.t` เพื่อดึง `(x, y, z)`
- `dist` = ระยะห่างระหว่าง `tip` กับ `M` → ใช้ `np.linalg.norm(tip - M)`

**From:** `q` ใช้ `q_goal` จากข้อ 1, ส่วน `M` ก็จากข้อ 1

**Expected outcome:** `tip ≈ M` และ `err ≤ 0.1` → แปลว่าคำตอบ IK ที่แก้มือ **ถูกต้อง**
(FK คือ "เฉลย" ที่เชื่อได้ ใช้ตรวจ IK ที่เราแก้มือ — ลองแกล้งเปลี่ยน `q_goal` ให้ผิดดู แล้ว `err` จะเด้งขึ้นทันที)

In [ ]:
q = q_goal

tip = None    # fill in
dist = None   # fill in

if dist is None:
    print('fill in tip and dist above')
else:
    print('tip:', np.round(tip, 4), '| M:', np.round(M, 4), '| err:', round(float(dist), 4))

### 3. let the toolbox solve IK (ikine), compare with q_goal
hint: robot.ikine_LM(SE3(*M), q0=[0,0,0], mask=[1,1,1,0,0,0], joint_limits=True); then use sol.q and sol.success

**Fill:** `sol` = ผลจากการเรียก `robot.ikine_LM(...)` (ตัวแก้ IK เชิงตัวเลขของ toolbox)

**Meaning arguement:**
- `SE3(*M)` = สร้าง target pose จากจุด `M` ข้อ 1
- `q0` = จุดเริ่มต้นที่ solver ใช้ไล่หา (initial guess)
- `mask=[1,1,1,0,0,0]` = ให้สนใจแค่ **ตำแหน่ง** (x,y,z) ไม่สนใจ **orientation** เพราะหุ่น **3-DOF** คุมทิศทางไม่ครบ
- `joint_limits=True` = เคารพลิมิตข้อต่อของหุ่น

**How to use result:** `sol.q` = คำตอบข้อต่อที่ toolbox หาได้, `sol.success` = ลู่เข้า (converge) สำเร็จไหม (True/False)

**หมายเหตุ:** `sol.q` อาจ **ไม่เท่ากับ** `q_goal` ที่แก้มือ แต่ก็ไปถึง `M` ได้เหมือนกัน เพราะเป็น **คนละ branch** (เช่น ศอกงอขึ้น/งอลง)

In [ ]:
sol = None    # fill in  (see hint above)

if sol is None:
    print('fill in the ikine_LM call')
else:
    print('toolbox q:', np.round(sol.q, 4), '| ok', sol.success)
    print('hand q   :', np.round(q_goal, 4))

### 4. check every branch
hint: robot.fkine(qb).t for each branch qb, compare to M

**Fill:** `branches` = ลิสต์ของ **ทุกคำตอบ IK (ทุก branch)** จาก Part A ข้อ 2/3 — แต่ละตัวเป็น `np.array` 3 ตัว

**Code:** วนเอาแต่ละ branch `qb` ไปเข้า FK (`robot.fkine(qb).t`) แล้ววัด error เทียบ `M`

**Expected outcome:** ทุก branch ที่ถูกต้องควรได้ `err ≈ 0` (≤ 0.1) — พิสูจน์ว่าหลายท่าที่ต่างกันก็เอื้อมถึงเป้าหมายเดียวกันได้

In [ ]:
branches = [
    np.array([0.0, 0.0, 0.0]),    # edit here : branch 1
    # np.array([0.0, 0.0, 0.0]),  # branch 2 if any
]

for i, qb in enumerate(branches):
    err = np.linalg.norm(robot.fkine(qb).t - M)
    print('branch', i + 1, np.round(qb, 4), '-> err', round(float(err), 4))